In [1]:
# ============================================================
# Pyomo MILP (Cluster, Non-Sequential) + Monthly Cycle Closure
# FINAL + Investment Requirements (min #stations / min #RB, and k >= x)
# + Gating P2P edges by RB build status (no bilinear terms)
# ============================================================

# (برای Colab لازم است؛ اگر محلی اجرا می‌کنی و نصب داری، می‌توانی این‌ها را حذف کنی)


import os, numpy as np, pandas as pd
from pyomo.environ import (
    ConcreteModel, Set, Param, Var, NonNegativeReals, Binary, Integers, Reals,
    Objective, Constraint, minimize, value, SolverFactory
)

# ------------------------ CONFIG ------------------------
DATA_XLSX_PATH = "p2p_ev_dataset_cluster_v2_common.xlsx"
RESULT_DIR = "opt_results_cluster_pyomo"
P2P_FULL = True

# ➊ الزام ساخت حداقل تعداد ایستگاه/باتری (قابل تنظیم)
REQUIRED_STATIONS = 3   # حداقل ایستگاه
REQUIRED_RB       = 1   # حداقل باتری مرکزی

# -------------------- Upload if needed --------------------
if not os.path.exists(DATA_XLSX_PATH):
    try:
        from google.colab import files
        print("⬆️ فایل p2p_ev_dataset_cluster_v2.xlsx را آپلود کنید ...")
        uploaded = files.upload()
        DATA_XLSX_PATH = list(uploaded.keys())[0]
    except Exception as e:
        raise FileNotFoundError("Dataset not found and upload failed.") from e

# -------------------- READ --------------------
xls = pd.ExcelFile(DATA_XLSX_PATH)
B = pd.read_excel(xls,"Sets_Buildings")["Building"].tolist()
V = pd.read_excel(xls,"Sets_EVs")["EV"].tolist()
S = pd.read_excel(xls,"Sets_Stations")["Station"].tolist()
R = pd.read_excel(xls,"Sets_RB")["RB"].tolist()
M = pd.read_excel(xls,"Sets_Months")["Month"].tolist()
T = pd.read_excel(xls,"Sets_Timeslots")["Timeslot"].tolist()

Time_Delta = pd.read_excel(xls,"Time_Delta").set_index(["Timeslot","Month"])
Prices     = pd.read_excel(xls,"Prices").set_index(["Timeslot","Month"])

Demand_B = pd.read_excel(xls,"Demand_Buildings").set_index(["Building","Timeslot","Month"])
Demand_EV= pd.read_excel(xls,"Demand_EVs").set_index(["EV","Timeslot","Month"])
Gen_B    = pd.read_excel(xls,"Generation_Buildings").set_index(["Building","Timeslot","Month"])

EV_params = pd.read_excel(xls,"EV_Params").set_index("EV")
EV_Avh    = pd.read_excel(xls,"EV_Availability_Hours").set_index(["EV","Station","Timeslot","Month"])

RB_params = pd.read_excel(xls,"RB_Params").set_index("RB")
RB_invest = pd.read_excel(xls,"RB_Invest").set_index("RB")

Station_params = pd.read_excel(xls,"Station_Params").set_index("Station")
System_costs   = pd.read_excel(xls,"System_Costs").set_index("Param")

GridBuy = pd.read_excel(xls,"Grid_BuyCaps").set_index(["Node","Timeslot","Month"])
GridSell= pd.read_excel(xls,"Grid_SellCaps").set_index(["Node","Timeslot","Month"])
Fbar    = pd.read_excel(xls,"P2P_Fbar").set_index(["i","j"])

Theta_RB = pd.read_excel(xls,"ThroughputCaps_RB").set_index(["RB","Month"]) if "ThroughputCaps_RB" in xls.sheet_names else None
Theta_EV = pd.read_excel(xls,"ThroughputCaps_EV").set_index(["EV","Month"]) if "ThroughputCaps_EV" in xls.sheet_names else None

# -------------------- STRICT NORMALIZATION (Hours_Available) --------------------
EV_Avh_df = EV_Avh.reset_index()   # EV, Station, Timeslot, Month, Frac_Available, Hours_Available
Delta_df  = Time_Delta.reset_index()  # Timeslot, Month, Delta_hours
EV_Avh_df = EV_Avh_df.merge(Delta_df, on=["Timeslot","Month"], how="left")

Hsum = EV_Avh_df.groupby(["EV","Timeslot","Month"], as_index=False)["Hours_Available"] \
                .sum().rename(columns={"Hours_Available":"Hsum"})
EV_Avh_df = EV_Avh_df.merge(Hsum, on=["EV","Timeslot","Month"], how="left")

eps = 1e-9
scale = np.where(EV_Avh_df["Hsum"]>0,
                 np.minimum(1.0, (EV_Avh_df["Delta_hours"]-eps)/EV_Avh_df["Hsum"]),
                 1.0)
EV_Avh_df["Hours_Available"] *= scale

# clamp نهایی
chk = EV_Avh_df.groupby(["EV","Timeslot","Month"], as_index=False)["Hours_Available"].sum()
chk = chk.merge(Delta_df, on=["Timeslot","Month"], how="left")
viol = chk[chk["Hours_Available"] > chk["Delta_hours"] + 1e-6]
if len(viol):
    for _, row in viol.iterrows():
        mask = (EV_Avh_df["EV"]==row["EV"]) & (EV_Avh_df["Timeslot"]==row["Timeslot"]) & (EV_Avh_df["Month"]==row["Month"])
        factor = (row["Delta_hours"]-eps) / row["Hours_Available"]
        EV_Avh_df.loc[mask, "Hours_Available"] *= factor

EV_Avh_scaled = EV_Avh_df.set_index(["EV","Station","Timeslot","Month"])
Havail_dict = EV_Avh_scaled["Hours_Available"].to_dict()

# -------------------- HELPERS --------------------
def dict_delta(): return {(t,m): float(Time_Delta.loc[(t,m),"Delta_hours"]) for t in T for m in M}
def dict_price_buy():  return {(t,m): float(Prices.loc[(t,m),"Pbuy"]) for t in T for m in M}
def dict_price_sell(): return {(t,m): float(Prices.loc[(t,m),"Psell"]) for t in T for m in M}

def dict_D():
    d={}
    for b in B:
        for t in T:
            for m in M:
                d[(b,t,m)] = float(Demand_B.loc[(b,t,m),"D_kWh"])
    for v in V:
        for t in T:
            for m in M:
                d[(v,t,m)] = 0.0
    for r in R:
        for t in T:
            for m in M:
                d[(r,t,m)] = 0.0
    return d

def dict_G():
    g={}
    for b in B:
        for t in T:
            for m in M:
                g[(b,t,m)] = float(Gen_B.loc[(b,t,m),"G_kWh"])
    for v in V:
        for t in T:
            for m in M:
                g[(v,t,m)] = 0.0
    for r in R:
        for t in T:
            for m in M:
                g[(r,t,m)] = 0.0
    return g

def dict_caps(df):
    return {idx: float(val) for idx,val in df.iloc[:,0].items()}

Pbuy_dict, Psell_dict, Delta_dict = dict_price_buy(), dict_price_sell(), dict_delta()
D_dict, G_dict = dict_D(), dict_G()
BuyCap_dict, SellCap_dict = dict_caps(GridBuy), dict_caps(GridSell)
Fcap_dict = {(i,j): float(Fbar.loc[(i,j),"Fcap_kWh"]) for (i,j) in Fbar.index}

def evp(col):
    for c in [col, col.replace("KWh","kWh"), col.replace("kWh","KWh")]:
        if c in EV_params.columns: return EV_params[c].to_dict()
    raise KeyError(col)

eta_chV  = evp("eta_ch")
eta_disV = evp("eta_dis")
EminV    = evp("Emin_kWh")
EmaxV    = evp("Emax_kWh")
PchV     = evp("Pch_max_kW")
PdisV    = evp("Pdis_max_kW")
CdegV    = evp("Cdeg_per_kWh")

eta_chR  = RB_params["eta_ch"].to_dict()
eta_disR = RB_params["eta_dis"].to_dict()
EminR    = RB_params["Emin_kWh"].to_dict()
EmaxR    = RB_params["Emax_KWh" if "Emax_KWh" in RB_params.columns else "Emax_kWh"].to_dict()
PchR     = RB_params["Pch_max_kW"].to_dict()
PdisR    = RB_params["Pdis_max_kW"].to_dict()
CdegR    = RB_params["Cdeg_per_kWh"].to_dict()
C_battery= RB_invest["C_battery"].to_dict()

Kmax   = Station_params["Kmax"].to_dict()
Pport  = Station_params["Pport_kW"].to_dict()
Usite  = Station_params["Usite_kW"].to_dict()
C_station     = Station_params["C_station"].to_dict()
C_per_charger = Station_params["C_per_charger"].to_dict()

C_curt = float(System_costs.loc["C_curt","Value"])

ThetaR_dict = {(r,m): float(Theta_RB.loc[(r,m),"Theta_R_kWh"]) for (r,m) in Theta_RB.index} if Theta_RB is not None else {}
ThetaV_dict = {(v,m): float(Theta_EV.loc[(v,m),"Theta_V_kWh"]) for (v,m) in Theta_EV.index} if Theta_EV is not None else {}

# -------------------- BUILD MODEL --------------------
def build_model():
    model = ConcreteModel()

    model.B = Set(initialize=B); model.V = Set(initialize=V)
    model.S = Set(initialize=S); model.R = Set(initialize=R)
    model.M = Set(initialize=M, ordered=True)
    model.T = Set(initialize=T, ordered=True)
    N = B + V + R
    model.N = Set(initialize=N)

    model.ITM  = Set(initialize=[(i,t,m) for i in N for t in T for m in M], dimen=3)
    if P2P_FULL:
        model.IJTM = Set(initialize=[(i,j,t,m) for i in N for j in N if i!=j for t in T for m in M], dimen=4)
    model.RTM  = Set(initialize=[(r,t,m) for r in R for t in T for m in M], dimen=3)
    model.VSTM = Set(initialize=[(v,s,t,m) for v in V for s in S for t in T for m in M], dimen=4)
    model.VTM  = Set(initialize=[(v,t,m) for v in V for t in T for m in M], dimen=3)

    model.Pbuy  = Param(model.T, model.M, initialize=Pbuy_dict,  within=Reals)
    model.Psell = Param(model.T, model.M, initialize=Psell_dict, within=Reals)
    model.Delta = Param(model.T, model.M, initialize=Delta_dict, within=Reals)

    model.D = Param(model.ITM, initialize=D_dict, within=Reals)
    model.G = Param(model.ITM, initialize=G_dict, within=Reals)

    model.BuyCap  = Param(model.ITM, initialize=BuyCap_dict, within=Reals)
    model.SellCap = Param(model.ITM, initialize=SellCap_dict, within=Reals)
    if P2P_FULL:
        model.Fcap = Param(model.N, model.N, initialize=lambda m,i,j: Fcap_dict.get((i,j),0.0), within=Reals)

    # Tech params
    model.eta_chV  = Param(model.V, initialize=eta_chV, within=Reals)
    model.eta_disV = Param(model.V, initialize=eta_disV, within=Reals)
    model.EminV    = Param(model.V, initialize=EminV, within=Reals)
    model.EmaxV    = Param(model.V, initialize=EmaxV, within=Reals)
    model.PchV     = Param(model.V, initialize=PchV, within=Reals)
    model.PdisV    = Param(model.V, initialize=PdisV, within=Reals)
    model.CdegV    = Param(model.V, initialize=CdegV, within=Reals)

    model.eta_chR  = Param(model.R, initialize=eta_chR, within=Reals)
    model.eta_disR = Param(model.R, initialize=eta_disR, within=Reals)
    model.EminR    = Param(model.R, initialize=EminR, within=Reals)
    model.EmaxR    = Param(model.R, initialize=EmaxR, within=Reals)
    model.PchR     = Param(model.R, initialize=PchR, within=Reals)
    model.PdisR    = Param(model.R, initialize=PdisR, within=Reals)
    model.CdegR    = Param(model.R, initialize=CdegR, within=Reals)

    # Investment & site
    model.C_station     = Param(model.S, initialize=C_station, within=Reals)
    model.C_per_charger = Param(model.S, initialize=C_per_charger, within=Reals)
    model.Kmax          = Param(model.S, initialize=Kmax, within=Reals)
    model.Pport         = Param(model.S, initialize=Pport, within=Reals)
    model.Usite         = Param(model.S, initialize=Usite, within=Reals)

    model.C_battery = Param(model.R, initialize=C_battery, within=Reals)
    model.C_curt    = Param(initialize=C_curt, within=Reals)

    # Availability (normalized)
    def H_init(m, v, s, t, mn): return float(Havail_dict.get((v,s,t,mn), 0.0))
    model.HoursAvail = Param(model.VSTM, initialize=H_init, within=Reals)

    if len(ThetaR_dict):
        model.ThetaR = Param(model.R, model.M, initialize=lambda m,r,mn: ThetaR_dict[(r,mn)], within=Reals)
    if len(ThetaV_dict):
        model.ThetaV = Param(model.V, model.M, initialize=lambda m,v,mn: ThetaV_dict[(v,mn)], within=Reals)

    # -------- Decision vars --------
    model.x = Var(model.S, within=Binary)                 # ساخت ایستگاه؟
    model.y = Var(model.R, within=Binary)                 # ساخت RB؟
    model.k = Var(model.S, within=Integers, bounds=(0,None))  # تعداد پورت‌ها

    model.Ebuy  = Var(model.ITM, within=NonNegativeReals)
    model.Esell = Var(model.ITM, within=NonNegativeReals)
    model.Lcurt = Var(model.ITM, within=NonNegativeReals)

    # ⬇️ ضد آربیتراژ خرید/فروش هم‌زمان از/به شبکه (برای هر گره-کلاستر-ماه)
    model.zBuySell = Var(model.ITM, within=Binary)

    if P2P_FULL:
        model.F = Var(model.IJTM, within=NonNegativeReals)

    model.CR = Var(model.RTM, within=NonNegativeReals)
    model.DR = Var(model.RTM, within=NonNegativeReals)

    model.CV = Var(model.VSTM, within=NonNegativeReals)
    model.DV = Var(model.VSTM, within=NonNegativeReals)

    # -------- Objective --------
    def obj_rule(m):
        invest = sum(m.C_station[s]*m.x[s] + m.C_per_charger[s]*m.k[s] for s in m.S) \
               + sum(m.C_battery[r]*m.y[r] for r in m.R)
        oper = sum(m.Pbuy[t,mn]*m.Ebuy[(i,t,mn)] - m.Psell[t,mn]*m.Esell[(i,t,mn)] + m.C_curt*m.Lcurt[(i,t,mn)]
                   for (i,t,mn) in m.ITM)
        degR = sum(m.CdegR[r]*(m.CR[(r,t,mn)] + m.DR[(r,t,mn)]) for (r,t,mn) in m.RTM)
        degV = sum(m.CdegV[v]*(sum(m.CV[(v,s,t,mn)] for s in m.S) + sum(m.DV[(v,s,t,mn)] for s in m.S))
                   for (v,t,mn) in m.VTM)
        return invest + oper + degR + degV
    model.TotalCost = Objective(rule=obj_rule, sense=minimize)

    # -------- Constraints --------
    # A) تعادل انرژی گره‌ها
    def balance_rule(m, i, t, mn):
        lhs = m.G[(i,t,mn)] + m.Ebuy[(i,t,mn)]
        rhs = m.D[(i,t,mn)] + m.Esell[(i,t,mn)] + m.Lcurt[(i,t,mn)]
        if P2P_FULL:
            lhs += sum(m.F[(j,i,t,mn)] for j in m.N if j!=i)
            rhs += sum(m.F[(i,j,t,mn)] for j in m.N if j!=i)
        if i in m.R:
            lhs += m.DR[(i,t,mn)]
            rhs += m.CR[(i,t,mn)]
        if i in m.V:
            lhs += sum(m.DV[(i,s,t,mn)] for s in m.S)
            rhs += sum(m.CV[(i,s,t,mn)] for s in m.S)
        return lhs == rhs
    model.Balance = Constraint(model.ITM, rule=balance_rule)

    # B) P2P ظرفیت
    if P2P_FULL:
        def fcap_rule(m, i, j, t, mn):
            return m.F[(i,j,t,mn)] <= m.Fcap[i,j] if (i != j) else Constraint.Skip
        model.FcapCons = Constraint(model.IJTM, rule=fcap_rule)

        # ================== NEW: GATE P2P BY RB BUILD STATUS (y) ==================
        # اگر مبدا RB باشد، خروجی از آن تنها وقتی مجاز است که ساخته شده باشد (y[i]=1)
        def F_gate_from_rule(m, i, j, t, mn):
            if i in m.R:
                return m.F[(i,j,t,mn)] <= m.Fcap[i,j] * m.y[i]
            return Constraint.Skip

        # اگر مقصد RB باشد، ورودی به آن تنها وقتی مجاز است که ساخته شده باشد (y[j]=1)
        def F_gate_to_rule(m, i, j, t, mn):
            if j in m.R:
                return m.F[(i,j,t,mn)] <= m.Fcap[i,j] * m.y[j]
            return Constraint.Skip

        model.F_gate_from = Constraint(model.IJTM, rule=F_gate_from_rule)
        model.F_gate_to   = Constraint(model.IJTM, rule=F_gate_to_rule)
        # ==========================================================================

    # C) شبکه
    def grid_buy_cap_rule(m, i, t, mn):
    # RB فقط اگر ساخته شده باشد می‌تواند از شبکه بخرد
        if i in m.R:
          return m.Ebuy[(i, t, mn)] <= m.BuyCap[(i, t, mn)] * m.y[i]
    # ساختمان و خودروی برقی می‌توانند مستقیم از شبکه بخرند (تا سقف ظرفیتشان)
        elif i in m.B or i in m.V:
          return m.Ebuy[(i, t, mn)] <= m.BuyCap[(i, t, mn)]
    # (در مدل فعلی S در ITM نیست؛ بنابراین به‌صورت پیش‌فرض صفر می‌ماند)
        else:
          return m.Ebuy[(i, t, mn)] == 0

    def grid_sell_cap_rule(m, i, t, mn):
    # RB فقط اگر ساخته شده باشد می‌تواند به شبکه بفروشد
        if i in m.R:
          return m.Esell[(i, t, mn)] <= m.SellCap[(i, t, mn)] * m.y[i]
    # ساختمان و خودروی برقی می‌توانند مستقیم به شبکه بفروشند (تا سقف ظرفیتشان)
        elif i in m.B or i in m.V:
          return m.Esell[(i, t, mn)] <= m.SellCap[(i, t, mn)]
        else:
          return m.Esell[(i, t, mn)] == 0

    model.GridBuyCap  = Constraint(model.ITM, rule=grid_buy_cap_rule)
    model.GridSellCap = Constraint(model.ITM, rule=grid_sell_cap_rule)

    # ضد خرید/فروش هم‌زمان در یک (i,t,m)
    def no_simul_buy_rule(m, i, t, mn):
        # اگر z=1 → خرید مجاز تا سقف BuyCap؛ اگر z=0 → Ebuy=0
        return m.Ebuy[i,t,mn]  <= m.BuyCap[i,t,mn]  * m.zBuySell[i,t,mn]

    def no_simul_sell_rule(m, i, t, mn):
        # اگر z=0 → فروش مجاز تا سقف SellCap؛ اگر z=1 → Esell=0
        return m.Esell[i,t,mn] <= m.SellCap[i,t,mn] * (1 - m.zBuySell[i,t,mn])

    model.NoSimulBuy  = Constraint(model.ITM, rule=no_simul_buy_rule)
    model.NoSimulSell = Constraint(model.ITM, rule=no_simul_sell_rule)

    # D) RB power->energy caps (خوشه‌ای)
    def rb_cap_ch_rule(m, r, t, mn):  return m.CR[(r,t,mn)] <= m.PchR[r]*m.Delta[t,mn]*m.y[r]
    def rb_cap_dis_rule(m, r, t, mn): return m.DR[(r,t,mn)] <= m.PdisR[r]*m.Delta[t,mn]*m.y[r]
    model.RB_CapCh  = Constraint(model.RTM, rule=rb_cap_ch_rule)
    model.RB_CapDis = Constraint(model.RTM, rule=rb_cap_dis_rule)

    # E) EV caps via Hours_Available
    def ev_cap_ch_rule(m, v, s, t, mn):  return m.CV[(v,s,t,mn)] <= m.PchV[v]*m.HoursAvail[(v,s,t,mn)]
    def ev_cap_dis_rule(m, v, s, t, mn): return m.DV[(v,s,t,mn)] <= m.PdisV[v]*m.HoursAvail[(v,s,t,mn)]
    model.EV_CapCh  = Constraint(model.VSTM, rule=ev_cap_ch_rule)
    model.EV_CapDis = Constraint(model.VSTM, rule=ev_cap_dis_rule)

    # F) قیود ایستگاه و لینک سرمایه‌گذاری
    def station_k_vs_x_rule(m, s): return m.k[s] <= m.Kmax[s]*m.x[s]
    model.Station_k_vs_x = Constraint(model.S, rule=station_k_vs_x_rule)

    def station_portcap_rule(m, s, t, mn):
        return sum(m.CV[(v,s,t,mn)] + m.DV[(v,s,t,mn)] for v in m.V) <= m.k[s]*m.Pport[s]*m.Delta[t,mn]
    model.Station_PortCap = Constraint([(s,t,mn) for s in S for t in T for mn in M], rule=station_portcap_rule)

    def station_sitecap_rule(m, s, t, mn):
        return sum(m.CV[(v,s,t,mn)] + m.DV[(v,s,t,mn)] for v in m.V) <= m.Usite[s]*m.Delta[t,mn]*m.x[s]
    model.Station_SiteCap = Constraint([(s,t,mn) for s in S for t in T for mn in M], rule=station_sitecap_rule)

    # G) چرخه‌بستهٔ ماهانه
    def rb_cycle_rule(m, r, mn):
        return sum(m.eta_chR[r]*m.CR[(r,t,mn)] for t in m.T) == sum((1.0/m.eta_disR[r])*m.DR[(r,t,mn)] for t in m.T)
    model.RB_Cycle = Constraint(model.R, model.M, rule=rb_cycle_rule)

    def ev_cycle_rule(m, v, mn):
        lhs = sum(m.eta_chV[v]*sum(m.CV[(v,s,t,mn)] for s in m.S) for t in m.T)
        rhs = sum((1.0/m.eta_disV[v])*sum(m.DV[(v,s,t,mn)] for s in m.S) for t in m.T)
        return lhs == rhs
    model.EV_Cycle = Constraint(model.V, model.M, rule=ev_cycle_rule)

    # H) سقف عبور انرژی ماهانه (اختیاری)
    if len(ThetaR_dict):
        def rb_throughput_rule(m, r, mn):
            return sum(m.CR[(r,t,mn)] + m.DR[(r,t,mn)] for t in m.T) <= m.ThetaR[r,mn]
        model.RB_Throughput = Constraint(model.R, model.M, rule=rb_throughput_rule)

    if len(ThetaV_dict):
        def ev_throughput_rule(m, v, mn):
            return sum(sum(m.CV[(v,s,t,mn)] + m.DV[(v,s,t,mn)] for s in m.S) for t in m.T) <= m.ThetaV[v,mn]
        model.EV_Throughput = Constraint(model.V, model.M, rule=ev_throughput_rule)

    # I) *** قیود اجباری احداث ***
    def min_stations_rule(m):
        return sum(m.x[s] for s in m.S) >= REQUIRED_STATIONS
    model.MinStations = Constraint(rule=min_stations_rule)

    def min_rbs_rule(m):
        return sum(m.y[r] for r in m.R) >= REQUIRED_RB
    model.MinRBs = Constraint(rule=min_rbs_rule)

    def k_lower_rule(m, s):
        return m.k[s] >= m.x[s]
    model.K_LowerBound = Constraint(model.S, rule=k_lower_rule)

    return model

model = build_model()

# -------------------- SOLVE --------------------
def get_solver():
    if SolverFactory('cbc').available(exception_flag=False):  return SolverFactory('cbc')
    if SolverFactory('glpk').available(exception_flag=False): return SolverFactory('glpk')
    if SolverFactory('highs').available(exception_flag=False):return SolverFactory('highs')
    raise RuntimeError("No MILP solver found. Please install CBC or GLPK.")

solver = get_solver()
res = solver.solve(model, tee=True)
print(res.solver.status, res.solver.termination_condition)
print("OBJ:", value(model.TotalCost))

# -------------------- EXPORT --------------------
os.makedirs(RESULT_DIR, exist_ok=True)

def dump_var(var, headers, fname):
    rows=[]
    for key in var:
        val = value(var[key])
        if val is None: continue
        rows.append(list(key)+[val] if isinstance(key, tuple) else [key, val])
    pd.DataFrame(rows, columns=headers+["value"]).to_csv(os.path.join(RESULT_DIR, fname), index=False)

# Investment
dump_var(model.x, ["Station"], "x_station.csv")
dump_var(model.k, ["Station"], "k_chargers.csv")
dump_var(model.y, ["RB"],      "y_RB.csv")

# Grid & Curtailment
dump_var(model.Ebuy,  ["Node","Timeslot","Month"], "E_grid_buy.csv")
dump_var(model.Esell, ["Node","Timeslot","Month"], "E_grid_sell.csv")
dump_var(model.Lcurt, ["Node","Timeslot","Month"], "Load_curt.csv")

# P2P
if P2P_FULL:
    dump_var(model.F, ["i","j","Timeslot","Month"], "F_p2p.csv")

# RB
dump_var(model.CR, ["RB","Timeslot","Month"], "C_RB.csv")
dump_var(model.DR, ["RB","Timeslot","Month"], "D_RB.csv")

# EV
dump_var(model.CV, ["EV","Station","Timeslot","Month"], "C_EV.csv")
dump_var(model.DV, ["EV","Station","Timeslot","Month"], "D_EV.csv")

print(f"✓ نتایج در پوشه `{RESULT_DIR}` ذخیره شد.")
# دانلود یکجا در کولب:
# from google.colab import files
# !zip -r opt_results_cluster_pyomo.zip opt_results_cluster_pyomo
# files.download('opt_results_cluster_pyomo.zip')


Running HiGHS 1.11.0 (git hash: 364c83a): Copyright (c) 2025 HiGHS under MIT licence terms
MIP  has 298226 rows; 248863 cols; 897907 nonzeros; 3247 integer variables (3235 binary)
Coefficient ranges:
  Matrix [9e-01, 1e+06]
  Cost   [1e+01, 1e+09]
  Bound  [1e+00, 1e+00]
  RHS    [1e+00, 1e+09]
Presolving model
56846 rows, 237119 cols, 586163 nonzeros  0s
56846 rows, 152159 cols, 416243 nonzeros  1s
56846 rows, 152159 cols, 416243 nonzeros  2s

Solving MIP model with:
   56846 rows
   152159 cols (3235 binary, 12 integer, 0 implied int., 148912 continuous, 0 domain fixed)
   416243 nonzeros

Src: B => Branching; C => Central rounding; F => Feasibility pump; J => Feasibility jump;
     H => Heuristic; L => Sub-MIP; P => Empty MIP; R => Randomized rounding; Z => ZI Round;
     I => Shifting; S => Solve LP; T => Evaluate node; U => Unbounded; X => User solution;
     z => Trivial zero; l => Trivial lower; u => Trivial upper; p => Trivial point

        Nodes      |    B&B Tree     |      